#### pip install jupyterlab ipykernel ipywidgets datasets torch torchvision torchaudio transformers peft evaluate 
pip install scikit-learn

In [1]:
import os
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    get_scheduler
)
from accelerate import Accelerator
from torch.optim import AdamW
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from tqdm.auto import tqdm

In [2]:

# Set up environment variables
os.environ['HF_HOME'] = 'd://cached'
os.environ['HF_DATASETS_CACHE'] = 'd://datasets_cache'
offload_dir = "d://offload_dir"
os.makedirs(offload_dir, exist_ok=True)

# Initialize accelerator
accelerator = Accelerator()

# Load dataset
dataset = load_dataset('shawhin/imdb-truncated')

# Model configuration
model_checkpoint = 'distilbert-base-uncased'
id2label = {0: "Negative", 1: "Positive"}
label2id = {"Negative": 0, "Positive": 1}

In [3]:
text_list = [
    "It was good.",
    "Not a fan, don't recommend.",
    "Better than the first one.",
    "This is not worth watching even once.",
    "This one is a pass."
]

In [4]:


# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=2, 
    id2label=id2label, 
    label2id=label2id
)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

# Add pad token if needed
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors=None,  # Important: Keep as None for dataset processing
        return_attention_mask=True
    )


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Tokenize datasets with label handling
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names  # Remove original columns
)

# Add labels back to the dataset
def add_labels(examples, indices):
    examples["labels"] = dataset["train"][indices]["label"]
    return examples

tokenized_dataset["train"] = tokenized_dataset["train"].map(
    add_labels,
    with_indices=True
)

tokenized_dataset["validation"] = tokenized_dataset["validation"].map(
    add_labels,
    with_indices=True
)

# Ensure dataset has the right format
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [6]:

# Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create dataloaders with proper batch handling
train_dataloader = DataLoader(
    tokenized_dataset["train"],
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_dataset["validation"],
    batch_size=8,
    collate_fn=data_collator
)

# LoRA Configuration
peft_config = LoraConfig(
    task_type="SEQ_CLS",
    r=4,
    lora_alpha=32,
    lora_dropout=0.01,
    target_modules=['q_lin']
)

In [7]:



# Apply LoRA to model
model = get_peft_model(model, peft_config)

# Training setup
optimizer = AdamW(model.parameters(), lr=1e-3)
num_epochs = 4
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

# Prepare everything with accelerator
model, optimizer, train_dataloader, eval_dataloader, lr_scheduler = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader, lr_scheduler
)


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    
    # Ensure labels are not None
    if labels is None:
        print("Warning: Labels are None!")
        return {"accuracy": 0.0, "f1": 0.0}
        
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "f1": float(f1_score(labels, predictions, average="weighted"))
    }

In [9]:
def show_predictions_before_training(text_list):
    """Show model predictions before fine-tuning"""
    print("Pre-training predictions:")
    print("-----------------------")
    model.eval()
    
    for text in text_list:
        # Tokenize single text input
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        
        with torch.no_grad():
            outputs = model(encoded["input_ids"])
            predictions = torch.argmax(outputs.logits, dim=-1)
            
        predicted_label = id2label[predictions.item()]
        print(f"{text} - {predicted_label}")
    print("\n")

In [10]:
def save_model(model, tokenizer, output_dir):
    """Save the model, tokenizer and configuration"""
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Get the unwrapped model if using accelerator
    unwrapped_model = accelerator.unwrap_model(model)
    
    # Save the PEFT model state dict and config
    unwrapped_model.save_pretrained(output_dir)
    
    # Save the tokenizer
    tokenizer.save_pretrained(output_dir)
    
    print(f"Model saved to {output_dir}")

In [11]:
def train():
    for epoch in range(num_epochs):
        model.train()
        progress_bar = tqdm(train_dataloader, desc=f"Training Epoch {epoch}")
        total_loss = 0
        best_accuracy = 0.0  # Start with 0 as the baseline
        
        for batch in progress_bar:
            # Ensure labels are present
            if "labels" not in batch:
                raise ValueError("Labels not found in batch!")
                
            with accelerator.accumulate(model):
                outputs = model(**batch)
                loss = outputs.loss
                total_loss += loss.item()
                accelerator.backward(loss)
                
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
                
                # Update progress bar description with loss
                progress_bar.set_postfix(loss=loss.item())
        
        # Evaluation
        model.eval()
        eval_metric = evaluate.load("accuracy")
        
        with tqdm(eval_dataloader, desc="Evaluating") as eval_progress:
            for batch in eval_progress:
                with torch.no_grad():
                    outputs = model(**batch)
                predictions = outputs.logits.argmax(dim=-1)
                eval_metric.add_batch(
                    predictions=accelerator.gather(predictions),
                    references=accelerator.gather(batch["labels"])
                )
        
        eval_results = eval_metric.compute()
        print(f"Epoch {epoch}: {eval_results}")
        print(f"Average loss: {total_loss / len(train_dataloader)}")
        
        # Save best model based on accuracy
        if eval_results["accuracy"] > best_accuracy:
            best_accuracy = eval_results["accuracy"]
            save_model(model, tokenizer, "d://models/sentiment_model")
            print(f"New best model saved with accuracy: {best_accuracy}")
        
        # Save final model at the end of each epoch
        save_model(model, tokenizer, "d://models/sentiment_model_final")

In [12]:


def predict(text_list):
    model.eval()
    print("Model predictions:")
    print("-----------------")
    for text in text_list:
        # Tokenize single text input
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"  # Use "pt" for single prediction
        )
        
        # Move to the correct device
        encoded = {k: v.to(accelerator.device) for k, v in encoded.items()}
        
        with torch.no_grad():
            outputs = model(**encoded)
            predictions = torch.argmax(outputs.logits, dim=-1)
            
        predicted_label = id2label[predictions.item()]
        print(f"{text} - {predicted_label}")


In [13]:
if __name__ == "__main__":
    # Example texts for testing
    text_list = [
        "It was good.",
        "Not a fan, don't recommend.",
        "Better than the first one.",
        "This is not worth watching even once.",
        "This one is a pass."
    ]
    
    # Show predictions before training
    show_predictions_before_training(text_list)
        
    # Run training
    train()
    
    # Make predictions
    predict(text_list)

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


Pre-training predictions:
-----------------------
It was good. - Negative
Not a fan, don't recommend. - Negative
Better than the first one. - Negative
This is not worth watching even once. - Negative
This one is a pass. - Negative




Training Epoch 0:   0%|          | 0/125 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 0: {'accuracy': 0.84}
Average loss: 0.4712517605423927
Model saved to d://models/sentiment_model
New best model saved with accuracy: 0.84
Model saved to d://models/sentiment_model_final


...

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000197C51BF9D0>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 70014e1d-ffbf-4211-8612-fe838a1e4440)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased/resolve/main/config.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000197C50F4190>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: bc12157c-4384-4ae4-b702-92a84ee84005)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.jso

Model saved to d://models/sentiment_model_final


Training Epoch 3:   0%|          | 0/125 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 3: {'accuracy': 0.897}
Average loss: 0.0892706765178591
Model saved to d://models/sentiment_model
New best model saved with accuracy: 0.897
Model saved to d://models/sentiment_model_final
Model predictions:
-----------------
It was good. - Positive
Not a fan, don't recommend. - Negative
Better than the first one. - Positive
This is not worth watching even once. - Negative
This one is a pass. - Positive
